# Label Refinement Pipeline

Iterative ink-label enhancement for Herculaneum scroll segments.

**Pipeline stages**
1. Filter pipeline — median denoising → CLAHE → morphological cleanup
2. Letter candidate extraction — connected components → confidence tiers → Greek template matching
3. LLM gap-filling — Claude resolves ambiguous candidates and infers missing characters
4. Label blend — α × LLM-updated map + (1-α) × original
5. Loop — iterate until convergence; optionally retrain SegmentInkNet on refined labels

Imports from: `src/label_filters.py`, `src/letter_candidates.py`, `src/llm_gap_fill.py`, `src/label_blend.py`, `src/refine_loop.py`

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import pipeline modules
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tifffile as tf

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.facecolor': '#111111',
    'figure.facecolor': '#1a1a1a',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': '#444444',
    'axes.titlecolor': 'white',
    'legend.facecolor': '#222222',
    'legend.edgecolor': '#555555',
})

from label_filters import load_label, apply_filters, adaptive_local_threshold, visualize_filter_stages
from letter_candidates import (
    GreekTemplateMatcher,
    extract_candidates,
    detect_text_lines,
    run_matching_pipeline,
    visualize_candidates_overlay,
    visualize_template_matches,
    visualize_text_lines,
    assign_confidence_tier,
    softmax_scores,
)
from llm_gap_fill import LLMGapFiller, visualize_llm_output, visualize_all_line_results
from label_blend import (
    blend_labels, compute_diff_stats, save_iteration_labels,
    visualize_diff, visualize_iteration_history, ALPHA_SCHEDULE,
)
from refine_loop import RefinementConfig, run_refinement

print('All modules loaded.')

## Configuration
Set `SEG_ID` and `LABEL_PATH` to your segment. `LABEL_PATH` can be:
- An existing pseudo-label (`ink_labels.tif`) to refine the training labels
- A model prediction (`{seg}_prob.npy`) to refine inference output

In [ ]:
# ─── Edit these ───────────────────────────────────────────────────────────
SEG_ID     = '20231221180251'          # smallest segment, used as val
LABEL_PATH = Path(f'../data/labelled_segments/{SEG_ID}/ink_labels.tif')
# LABEL_PATH = Path(f'../predictions/{SEG_ID}_prob.npy')  # use model output instead

THRESHOLD   = 0.40   # binarisation threshold
MIN_AREA    = 50     # minimum CC area (pixels)
MAX_AREA    = 8000   # maximum CC area
CROP        = None   # set to (y0, y1, x0, x1) to inspect a subregion
# CROP      = (200, 600, 100, 900)  # example crop
# ──────────────────────────────────────────────────────────────────────────

print(f'Segment : {SEG_ID}')
print(f'Label   : {LABEL_PATH}')
print(f'Exists  : {LABEL_PATH.exists()}')

## 1. Load & Display Raw Input

In [ ]:
prob_orig = load_label(LABEL_PATH)
print(f'Shape: {prob_orig.shape}   min={prob_orig.min():.3f}  max={prob_orig.max():.3f}  '
      f'mean={prob_orig.mean():.4f}  ink%={(prob_orig > THRESHOLD).mean()*100:.1f}%')

display = prob_orig[CROP[0]:CROP[1], CROP[2]:CROP[3]] if CROP else prob_orig

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Raw input — {SEG_ID}', fontsize=12, fontweight='bold')

axes[0].imshow(display, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[0].set_title('Probability map'); axes[0].axis('off')

axes[1].imshow(display > THRESHOLD, cmap='gray', interpolation='nearest')
axes[1].set_title(f'Binarised  (t={THRESHOLD})'); axes[1].axis('off')

axes[2].hist(prob_orig.flatten(), bins=80, range=(0, 1), color='steelblue',
             edgecolor='none', alpha=0.9)
axes[2].axvline(THRESHOLD, color='tomato', lw=1.5, label=f'threshold={THRESHOLD}')
axes[2].set_xlabel('Probability'); axes[2].set_ylabel('Pixel count')
axes[2].set_title('Pixel distribution'); axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Stage 1 — Filter Pipeline
Median denoising → CLAHE contrast normalisation → morphological stroke repair.

In [ ]:
prob_filtered = apply_filters(
    prob_orig,
    median_radius=1,
    clahe_clip=0.02,
    clahe_tile=8,
    close_disk=2,
    open_disk=1,
    threshold=THRESHOLD,
)

ink_before = (prob_orig   > THRESHOLD).mean() * 100
ink_after  = (prob_filtered > THRESHOLD).mean() * 100
print(f'Ink coverage  before: {ink_before:.2f}%   after: {ink_after:.2f}%  '
      f'delta: {ink_after - ink_before:+.2f}%')

fig = visualize_filter_stages(
    prob_orig, prob_filtered,
    crop=CROP,
    title_suffix=f'  — {SEG_ID}',
)
plt.show()

# Optional: compare global vs adaptive threshold
prob_adaptive = adaptive_local_threshold(prob_filtered, window=31, offset=0.05)
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('Global vs adaptive threshold (on filtered map)', fontsize=11)
d = lambda a: a[CROP[0]:CROP[1], CROP[2]:CROP[3]] if CROP else a
axes2[0].imshow(d(prob_filtered) > THRESHOLD, cmap='gray')
axes2[0].set_title(f'Global t={THRESHOLD}'); axes2[0].axis('off')
axes2[1].imshow(d(prob_adaptive), cmap='gray')
axes2[1].set_title('Adaptive (local mean + 0.05)'); axes2[1].axis('off')
plt.tight_layout(); plt.show()

## 3. Stage 2 — Letter Candidate Extraction
Connected components → confidence tiers → Greek template matching (multi-style, multi-scale, Chamfer + NCC + IoU + Hu moments).

In [ ]:
# Build template bank once — reuse across all cells
print('Building template bank (3 styles × 3 sizes × 24 letters)…')
matcher = GreekTemplateMatcher()
print('Done.')

# Extract candidates from filtered map
candidates = extract_candidates(
    prob_filtered,
    threshold=THRESHOLD,
    min_area=MIN_AREA,
    max_area=MAX_AREA,
)
print(f'\n{len(candidates)} candidates extracted')

# Confidence tier breakdown
tier_counts = {'HIGH': 0, 'MEDIUM': 0, 'LOW': 0}
for c in candidates:
    tier_counts[assign_confidence_tier(c)] += 1
for t, n in tier_counts.items():
    print(f'  {t:6s}: {n:5d}  ({n/len(candidates)*100:.1f}%)')

# Run template matching on HIGH + MEDIUM
print('\nRunning template matching…')
for i, cand in enumerate(candidates):
    if assign_confidence_tier(cand) != 'LOW':
        cand.matches = matcher.match(cand.patch, top_k=3)
    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(candidates)}')
print('Done.')

### 3a. Candidate overlay

In [ ]:
fig = visualize_candidates_overlay(
    prob_filtered, candidates,
    crop=CROP,
    figsize=(14, 9),
    max_labels=400,
)
plt.show()

### 3b. Template match inspection
Each row: candidate patch | top-3 matching Greek letter templates with scores.

In [ ]:
fig = visualize_template_matches(
    candidates, matcher,
    n=24,
    figsize=(16, 22),
)
plt.show()

### 3c. Template reference sheet
All 24 Greek uppercase letters rendered in 3 styles used by the matcher.

In [ ]:
from letter_candidates import GREEK_UPPER, GREEK_NAMES

styles  = ['serif', 'uncial', 'skeleton']
size_px = 64

fig, axes = plt.subplots(len(styles), len(GREEK_UPPER), figsize=(20, 4))
fig.suptitle('Template bank — 3 styles × 24 letters', fontsize=11)

for row, style in enumerate(styles):
    for col, ch in enumerate(GREEK_UPPER):
        tpl = matcher._bank[size_px][style][ch][0]
        axes[row, col].imshow(tpl, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(ch, fontsize=8)
    axes[row, 0].set_ylabel(style, fontsize=9, rotation=0, ha='right', va='center')

plt.tight_layout(rect=(0.03, 0, 1, 0.96))
plt.show()

## 4. Text Line Detection

In [ ]:
lines = detect_text_lines(candidates)
print(f'{len(lines)} lines detected')

# Per-line stats
lengths = [len(l.candidates) for l in lines]
print(f'Candidates per line — min:{min(lengths)}  median:{int(np.median(lengths))}  max:{max(lengths)}')

fig = visualize_text_lines(prob_filtered, lines, crop=CROP, figsize=(14, 9))
plt.show()

# Histogram of candidates per line
fig2, ax2 = plt.subplots(figsize=(9, 3))
ax2.bar(range(len(lengths)), lengths, color='steelblue', edgecolor='none', alpha=0.8)
ax2.set_xlabel('Line index'); ax2.set_ylabel('Candidates in line')
ax2.set_title('Candidates per text line'); plt.tight_layout(); plt.show()

### Line detail — inspect one line
Print the LLM context string that will be sent to Claude for a chosen line.

In [ ]:
from letter_candidates import build_llm_line_context

LINE_IDX = 5   # change this to inspect a different line

if LINE_IDX < len(lines):
    line = lines[LINE_IDX]
    ctx  = build_llm_line_context(line, prob_filtered.shape[1])
    print(f'Line {line.line_id}  ({len(line.candidates)} candidates)\n')
    print(ctx)

    # Show top-5 match detail for each HIGH/MEDIUM candidate
    print('\n--- Detailed match scores ---')
    for cand in line.candidates[:10]:
        tier = assign_confidence_tier(cand)
        if tier == 'LOW' or not cand.matches:
            continue
        probs = softmax_scores(cand.matches)
        top3  = '  '.join(f'{ch} {p:.2f}' for ch, p in probs[:3])
        print(f'  {tier:6s}  prob={cand.mean_prob:.2f}  →  {top3}')
        print(f'            Chamfer={cand.matches[0].chamfer:.3f}  '
              f'NCC={cand.matches[0].ncc:.3f}  IoU={cand.matches[0].iou:.3f}  '
              f'Hu={cand.matches[0].hu_dist:.3f}  '
              f'style={cand.matches[0].best_style}  sz={cand.matches[0].best_size}')

## 5. Stage 3 — LLM Gap-Filling (single line demo)
Requires `ANTHROPIC_API_KEY` in the environment.

In [ ]:
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY not set — skipping LLM cell.')
    print('Set it with: import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."')
else:
    filler = LLMGapFiller(model='claude-opus-4-7')

    # Demo: run on a few lines around the chosen line index
    demo_lines = [lines[i] for i in range(
        max(0, LINE_IDX - 1), min(len(lines), LINE_IDX + 3)
    )]

    H, W = prob_filtered.shape
    fill_results = filler.fill_lines(
        demo_lines, W, H, batch_size=3, skip_all_high=True
    )

    print(f'\n{len(fill_results)} lines processed by LLM')
    for r in fill_results:
        if r.error:
            print(f'  Line {r.line_id}: ERROR — {r.error}')
        else:
            chars = ' '.join(f'{p.char}({p.source[:3]})' for p in r.predictions[:12])
            print(f'  Line {r.line_id}: {chars}')

### LLM output visualisation

In [ ]:
if os.environ.get('ANTHROPIC_API_KEY') and 'fill_results' in dir():
    result_by_id = {r.line_id: r for r in fill_results}

    for demo_line in demo_lines[:3]:
        fr = result_by_id.get(demo_line.line_id)
        if fr is None:
            continue
        fig = visualize_llm_output(prob_filtered, demo_line, fr, figsize=(18, 4))
        plt.show()

        if not fr.error and fr.raw_llm_response:
            print(f'Claude raw response (line {fr.line_id}):')
            print(fr.raw_llm_response[:400])
            print()

## 6. One Full Iteration — Filters + LLM + Blend
Runs the complete pipeline for a single iteration and shows the before/after diff.

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('Skipping — ANTHROPIC_API_KEY not set.')
else:
    H, W = prob_orig.shape
    ALPHA = ALPHA_SCHEDULE[0]   # first iteration blend weight

    # ── Stage 3: LLM on all lines ──────────────────────────────────────────
    print('Running LLM gap-filling on all lines…')
    all_fill_results = filler.fill_lines(lines, W, H, batch_size=5)

    n_errs    = sum(1 for r in all_fill_results if r.error)
    n_preds   = sum(len(r.predictions) for r in all_fill_results)
    n_gaps    = sum(1 for r in all_fill_results for p in r.predictions if p.source == 'llm_gap')
    n_med     = sum(1 for r in all_fill_results for p in r.predictions if p.source == 'llm_med')
    print(f'  {len(all_fill_results)} lines  |  {n_preds} total predictions  '
          f'({n_med} llm_med, {n_gaps} llm_gap, {n_errs} errors)')

    # ── Stage 3 → pixel update ─────────────────────────────────────────────
    prob_llm_updated = filler.apply_to_prob_map(prob_filtered, all_fill_results)

    # ── Stage 4: Blend ─────────────────────────────────────────────────────
    prob_blended = blend_labels(prob_orig, prob_llm_updated, alpha=ALPHA)

    stats = compute_diff_stats(prob_orig, prob_blended)
    print(f'  mean|Δ|={stats["mean_abs_diff"]:.4f}  '
          f'+{stats["pixels_increased_pct"]:.1f}%  '
          f'-{stats["pixels_decreased_pct"]:.1f}%')

### Diff visualisation

In [ ]:
if os.environ.get('ANTHROPIC_API_KEY') and 'prob_blended' in dir():
    fig = visualize_diff(
        prob_orig, prob_llm_updated, prob_blended,
        crop=CROP,
        iteration=1, alpha=ALPHA,
        figsize=(18, 10),
    )
    plt.show()

    # Side-by-side scroll through a horizontal strip
    if CROP is None:
        H, W = prob_orig.shape
        strip_y0, strip_y1 = H // 4, H // 4 + 200
    else:
        strip_y0, strip_y1 = CROP[0], CROP[0] + 200

    fig2, axes2 = plt.subplots(3, 1, figsize=(18, 7))
    fig2.suptitle('Horizontal strip comparison (200 px tall)', fontsize=11)
    for ax, arr, title in zip(
        axes2,
        [prob_orig, prob_llm_updated, prob_blended],
        ['Original labels', 'LLM-updated map', f'Blended (α={ALPHA})'],
    ):
        ax.imshow(arr[strip_y0:strip_y1], cmap='gray', vmin=0, vmax=1,
                  aspect='auto', interpolation='nearest')
        ax.set_title(title, fontsize=10); ax.axis('off')
    plt.tight_layout(); plt.show()

## 7. Full Iterative Refinement Loop
Runs multiple iterations until convergence. Set `MAX_ITER` to control depth.

> **Note**: each iteration calls the Claude API (~100 requests/segment). Estimated cost: ~$0.50–$1.50 per segment per iteration at Opus 4.7 pricing.

In [ ]:
MAX_ITER  = 3
RUN_LOOP  = bool(os.environ.get('ANTHROPIC_API_KEY'))  # skip if no key

if not RUN_LOOP:
    print('Skipping — ANTHROPIC_API_KEY not set.')
else:
    cfg = RefinementConfig(
        seg_id       = SEG_ID,
        label_path   = LABEL_PATH,
        mode         = 'label_only',   # change to 'full_loop' to include retraining
        max_iterations = MAX_ITER,
        threshold    = THRESHOLD,
        min_area     = MIN_AREA,
        max_area     = MAX_AREA,
        out_dir      = Path('../predictions/refinement'),
        log_path     = Path('../models/refinement_log.json'),
    )

    final_labels, history = run_refinement(
        cfg,
        matcher=matcher,  # reuse the one we already built
    )

    print(f'\nFinal ink coverage: {(final_labels > THRESHOLD).mean()*100:.2f}%')

### Convergence plot

In [ ]:
log_path = Path('../models/refinement_log.json')
if log_path.exists():
    fig = visualize_iteration_history(log_path, figsize=(13, 5))
    if fig:
        plt.show()
else:
    print('No log file found — run the loop first.')

## 8. Final Comparison — Original vs Refined

In [ ]:
# Load the latest refined labels if loop was run, else use one-iteration output
refinement_dir = Path('../predictions/refinement')
refined_files  = sorted(refinement_dir.glob(f'{SEG_ID}_labels_v*.tif'))

if refined_files:
    latest = refined_files[-1]
    print(f'Loading refined labels from: {latest}')
    final_labels = load_label(latest)

    d = lambda a: a[CROP[0]:CROP[1], CROP[2]:CROP[3]] if CROP else a

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    fig.suptitle(f'Original vs Refined labels — {SEG_ID}', fontsize=12, fontweight='bold')

    axes[0].imshow(d(prob_orig), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    axes[0].set_title('Original labels'); axes[0].axis('off')

    axes[1].imshow(d(final_labels), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    v = latest.stem.split('_v')[-1]
    axes[1].set_title(f'Refined labels (v{v})'); axes[1].axis('off')

    abs_diff = np.abs(d(final_labels) - d(prob_orig))
    im = axes[2].imshow(abs_diff, cmap='hot', vmin=0, vmax=0.3, interpolation='nearest')
    axes[2].set_title('|refined − original|'); axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout(); plt.show()

    stats = compute_diff_stats(prob_orig, final_labels)
    print(f'Overall change: mean|Δ|={stats["mean_abs_diff"]:.4f}  '
          f'+{stats["pixels_increased_pct"]:.1f}%  '
          f'-{stats["pixels_decreased_pct"]:.1f}%')
else:
    print(f'No refined labels found in {refinement_dir}.')
    print('Run sections 6 or 7 first.')

## 9. Save Final Refined Labels
Writes the final labels to `data/labelled_segments/{seg_id}/ink_labels_refined.tif` so they can be used as training labels in `segment_ink_detection.ipynb`.

In [ ]:
if 'final_labels' in dir() and refined_files:
    out_path = Path(f'../data/labelled_segments/{SEG_ID}/ink_labels_refined.tif')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tf.imwrite(str(out_path), (final_labels * 255).astype('uint8'))
    print(f'Saved refined labels → {out_path}')
    print(f'Shape: {final_labels.shape}  ink%={(final_labels > THRESHOLD).mean()*100:.2f}%')
    print()
    print('To use refined labels in segment training, update CFG in segment_ink_detection.ipynb:')
    print(f'  label_file = "ink_labels_refined.tif"')
else:
    print('Run sections 7 or 8 first to generate refined labels.')